In [1]:
from pyspark.sql import functions as F

# Read from Bronze
df_bronze = spark.read.format("delta").table("bronze.vessels")

# Deduplicate - same ship same timestamp
df_dedup = df_bronze.dropDuplicates(["mmsi", "timestamp"])

# Filter bad data
df_clean = df_dedup.filter(
    (F.col("latitude").isNotNull()) &
    (F.col("longitude").isNotNull()) &
    (F.col("latitude").between(45.20, 45.55)) &
    (F.col("longitude").between(12.20, 12.55)) &
    (F.col("sog") >= 0) &
    (F.col("sog") < 50)
)

# Add violation flag
df_silver = df_clean.withColumn(
    "is_violation",
    F.when(F.col("sog") > 5.0, True).otherwise(False)
).withColumn(
    "vessel_category",
    F.when(F.col("ship_type_code").between(60, 69), "Passenger")
     .when(F.col("ship_type_code").between(70, 79), "Cargo")
     .when(F.col("ship_type_code").between(80, 89), "Tanker")
     .when(F.col("ship_type_code") == 52, "Tug")
     .when(F.col("ship_type_code") == 30, "Fishing")
     .when(F.col("ship_type_code") == 37, "Pleasure craft")
     .otherwise("Other")
).withColumn(
    "processed_at",
    F.current_timestamp()
)

# Write to Silver
df_silver.write.format("delta").mode("append").saveAsTable("silver.vessels_clean")

print(f"Silver processed: {df_silver.count()} rows written")

StatementMeta(, a13f448f-0e67-4173-a203-30b392c26be2, 3, Finished, Available, Finished, False)

Silver processed: 414 rows written
